# Chemical Metadata Enrichment via PubChem

Takes a CSV with chemical names, matches them against the **PubChem** database, and outputs an enriched CSV with: PubChem CID, SMILES, molecular formula, InChIKey, and synonyms.

**How it works:**
- Downloads PubChem's bulk FTP files (CID-Title, CID-Synonym-filtered, CID-SMILES) for offline matching
- Scans titles first for exact matches, then falls back to synonym matching
- Fills SMILES from the CID-SMILES file
- For matched compounds, calls the **PubChem REST API** to fetch molecular formula, InChIKey, and up to 10 deduplicated synonyms

**Output columns:** `preferred_name`, `synonyms`, `pubchem_cid`, `chebi_id`, `inchikey`, `smiles`, `molecular_formula`, `source_db`, `source_url`

**Rerun-safe:** Re-running the API enrichment step preserves previously fetched fields — only unmatched rows are re-queried. Cleanup of PubChem FTP files happens only at the final save.

In [ ]:
!pip install pandas requests -q

In [ ]:
import os, pandas as pd
from google.colab import files

def read_csv_safe(path):
    for enc in ['utf-8', 'cp1252', 'latin-1']:
        try: return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError: continue
    return pd.read_csv(path, encoding='utf-8', errors='replace')

# Upload your chemical.csv file
uploaded = files.upload()
input_csv = list(uploaded.keys())[0]
print(f'Using: {input_csv}')
df = read_csv_safe(input_csv)
chem_col = 'chemical' if 'chemical' in df.columns else df.columns[0]
unique_chems = {str(x).strip().lower():str(x).strip() for x in df[chem_col].dropna()}
print(f'Total: {len(unique_chems)} chemicals')

## Download PubChem files

In [ ]:
import urllib.request, os
os.makedirs('pubchem', exist_ok=True)
files_list = [
    ('CID-Title.gz', 'https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/CID-Title.gz'),
    ('CID-Synonym-filtered.gz', 'https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/CID-Synonym-filtered.gz'),
    ('CID-SMILES.gz', 'https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/CID-SMILES.gz')
]
for fname, url in files_list:
    if not os.path.exists('pubchem/'+fname):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(url, 'pubchem/'+fname)
print('FTP files downloaded!')

## Scan files sequentially

In [ ]:
import gzip

# Store only YOUR chemicals
results = {}
for c in unique_chems:
    results[c] = {'orig': unique_chems[c], 'status': 'unmatched', 'cid': ''}

unmatched = set(results.keys())
print(f'Starting with {len(unmatched)} unmatched')

# Scan Title
print('Scanning Title...')
with gzip.open('pubchem/CID-Title.gz', 'rt') as f:
    count = 0
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            name = parts[1].lower().strip()
            if name in unmatched:
                results[name]['cid'] = parts[0]
                results[name]['title'] = parts[1]
                results[name]['status'] = 'exact'
                results[name]['src'] = 'title'
        count += 1
        if count % 10000000 == 0:
            print(f'  {count}...', end='\r')

matched_now = sum(1 for r in results.values() if r['status'] != 'unmatched')
print(f'Title done: {matched_now} matched, {len(unmatched)-matched_now} remaining')

In [ ]:
# Scan Synonyms
unmatched = set(k for k,v in results.items() if v['status'] == 'unmatched')
print(f'Scanning Synonyms for {len(unmatched)}...')
with gzip.open('pubchem/CID-Synonym-filtered.gz', 'rt') as f:
    count = 0
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            cid = parts[0]
            for syn in parts[1].split('|')[:30]:
                syn = syn.lower().strip()
                if syn and syn in unmatched and results[syn]['status'] == 'unmatched':
                    results[syn]['cid'] = cid
                    results[syn]['status'] = 'synonym'
                    results[syn]['matched_synonym'] = syn
        count += 1
        if count % 20000000 == 0:
            print(f'  {count}...', end='\r')

matched = sum(1 for r in results.values() if r['status'] != 'unmatched')
print(f'Synonyms done: {matched} total matched')

In [ ]:
# FTP phase only uses synonym bulk file for CID matching above.
# Human-readable synonym export will be filled later via API.
print('Skipping FTP synonym export pool; synonyms will be fetched by API later.')

In [ ]:
# Get FTP bulk fields
matched_cids = set(r['cid'] for r in results.values() if r['cid'])
print(f'Getting props for {len(matched_cids)} CIDs...')

# SMILES
with gzip.open('pubchem/CID-SMILES.gz', 'rt') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 2 and parts[0] in matched_cids:
            for k,v in results.items():
                if v.get('cid') == parts[0]:
                    v['smiles'] = parts[1]
print('SMILES done')

In [ ]:
# Do not clean up PubChem files here. In Colab, early cleanup makes reruns brittle.
print('PubChem files retained for notebook safety. Cleanup happens after final save.')

## Step 4: Build output

In [ ]:
OUTPUT = ['preferred_name','synonyms','pubchem_cid','inchikey','smiles','molecular_formula','source_db','source_url']

preserved_api_by_cid = {}
preserved_api_by_row = {}
if 'res_df' in globals() and isinstance(res_df, pd.DataFrame):
    for row in res_df.to_dict('records'):
        cid = str(row.get('pubchem_cid', '')).strip()
        preferred_name = str(row.get('preferred_name', '')).strip()
        payload = {
            'synonyms': row.get('synonyms', ''),
            'inchikey': row.get('inchikey', ''),
            'molecular_formula': row.get('molecular_formula', '')
        }
        if cid:
            preserved_api_by_cid.setdefault(cid, payload)
        preserved_api_by_row[(preferred_name, cid)] = payload

rows = []
for k, v in results.items():
    r = {c:'' for c in OUTPUT}
    input_name = v.get('orig', k)
    r['preferred_name'] = v.get('title', input_name)
    cid = v.get('cid', '')
    r['pubchem_cid'] = cid
    r['synonyms'] = ''
    r['inchikey'] = ''
    r['smiles'] = v.get('smiles', '')
    r['molecular_formula'] = ''

    if r['pubchem_cid']:
        r['source_db'] = 'PubChem'
        r['source_url'] = 'https://pubchem.ncbi.nlm.nih.gov/compound/' + r['pubchem_cid']

    preserved = preserved_api_by_row.get((r['preferred_name'], r['pubchem_cid']))
    if preserved is None and r['pubchem_cid']:
        preserved = preserved_api_by_cid.get(r['pubchem_cid'], {})
    if preserved:
        r['synonyms'] = preserved.get('synonyms', r['synonyms'])
        r['inchikey'] = preserved.get('inchikey', r['inchikey'])
        r['molecular_formula'] = preserved.get('molecular_formula', r['molecular_formula'])

    rows.append(r)

res_df = pd.DataFrame(rows, columns=OUTPUT)
matched = sum(1 for v in results.values() if v.get('cid'))
matched_df = res_df[res_df['pubchem_cid'].fillna('').astype(str).str.strip() != '']
smiles_filled = sum(matched_df['smiles'].fillna('').astype(str).str.strip() != '')
smiles_missing = matched - smiles_filled
print(f'Total: {len(res_df)}, Matched: {matched}')
print(f'SMILES filled from FTP: {smiles_filled}/{matched}')
if matched and smiles_filled == 0:
    raise RuntimeError('FTP SMILES enrichment is empty for all matched rows. Re-run Step 3 top-to-bottom before saving.')
if matched and smiles_missing:
    print(f'WARNING: {smiles_missing} matched rows still have blank SMILES after FTP enrichment.')
print('Step 4 is rerun-safe: existing API-enriched fields are preserved when possible.')

## OPTIONAL: Save CSV after FTP stage (before API enrichment)

In [ ]:
# Save a true FTP-stage CSV even if API enrichment has already been run in this kernel
ftp_stage_df = res_df.copy()
for col in ['synonyms', 'inchikey', 'molecular_formula']:
    ftp_stage_df[col] = ''
ftp_stage_df.to_csv('chemical_enriched_after_ftp.csv', index=False)
print('Saved: chemical_enriched_after_ftp.csv')
matched = sum(1 for v in results.values() if v.get('cid'))
print(f'Matched: {matched}, Unmatched: {len(ftp_stage_df) - matched}')
files.download('chemical_enriched_after_ftp.csv')
print('You can download this FTP-stage file now before API enrichment.')

## Step 5: Fill synonyms, InChIKey, and molecular formula via API

In [ ]:
import requests, time, re
to_fill = res_df[res_df['pubchem_cid'] != ''].copy()
print(f'API enrich for {len(to_fill)} matched chemicals')
inchikey_pattern = re.compile(r'^[A-Z]{14}-[A-Z]{10}-[A-Z]$')
property_batch_size = 100
synonym_batch_size = 25
unique_cids = [str(cid) for cid in to_fill['pubchem_cid'].dropna().astype(str).unique() if str(cid).strip()]
cid_to_indexes = {}
for idx, row in to_fill.iterrows():
    cid = str(row['pubchem_cid']).strip()
    cid_to_indexes.setdefault(cid, []).append(idx)

session = requests.Session()
cid_to_formula = {}
cid_to_inchikey = {}
cid_to_synonym_pool = {}
failed_property_batches = []
failed_synonym_batches = []

def chunked(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]

for batch_no, batch in enumerate(chunked(unique_cids, property_batch_size), start=1):
    returned = set()
    try:
        cid_str = ','.join(batch)
        prop_url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_str}/property/MolecularFormula,InChIKey/JSON'
        r = session.get(prop_url, timeout=20)
        if r.status_code == 200:
            props = r.json().get('PropertyTable', {}).get('Properties', [])
            for prop in props:
                cid = str(prop.get('CID', '')).strip()
                if not cid:
                    continue
                returned.add(cid)
                cid_to_formula[cid] = prop.get('MolecularFormula', '')
                key = prop.get('InChIKey', '')
                if inchikey_pattern.fullmatch(key):
                    cid_to_inchikey[cid] = key
        else:
            failed_property_batches.append((batch, f'HTTP {r.status_code}'))
    except Exception as e:
        failed_property_batches.append((batch, str(e)))
    missing = [cid for cid in batch if cid not in returned]
    if missing:
        failed_property_batches.append((missing, 'missing in batch response'))
    time.sleep(0.1)
    if batch_no % 10 == 0:
        print(f'Property batches: {min(batch_no * property_batch_size, len(unique_cids))}/{len(unique_cids)}', end='\r')

property_retry_cids = sorted({cid for batch, _ in failed_property_batches for cid in batch})
for cid in property_retry_cids:
    try:
        prop_url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/MolecularFormula,InChIKey/JSON'
        r = session.get(prop_url, timeout=20)
        if r.status_code == 200:
            props = r.json().get('PropertyTable', {}).get('Properties', [])
            if props:
                cid_to_formula[cid] = props[0].get('MolecularFormula', '')
                key = props[0].get('InChIKey', '')
                if inchikey_pattern.fullmatch(key):
                    cid_to_inchikey[cid] = key
    except Exception:
        pass
    time.sleep(0.05)

for batch_no, batch in enumerate(chunked(unique_cids, synonym_batch_size), start=1):
    returned = set()
    try:
        cid_str = ','.join(batch)
        syn_url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_str}/synonyms/JSON'
        r = session.get(syn_url, timeout=20)
        if r.status_code == 200:
            info_list = r.json().get('InformationList', {}).get('Information', [])
            for info in info_list:
                cid = str(info.get('CID', '')).strip()
                if not cid:
                    continue
                returned.add(cid)
                seen = set()
                cleaned = []
                for syn in info.get('Synonym', []):
                    syn = syn.strip()
                    syn_norm = syn.lower()
                    if not syn:
                        continue
                    if syn_norm.startswith('inchi='):
                        continue
                    if inchikey_pattern.fullmatch(syn):
                        continue
                    if syn_norm in seen:
                        continue
                    seen.add(syn_norm)
                    cleaned.append(syn)
                    if len(cleaned) == 10:
                        break
                cid_to_synonym_pool[cid] = cleaned
        else:
            failed_synonym_batches.append((batch, f'HTTP {r.status_code}'))
    except Exception as e:
        failed_synonym_batches.append((batch, str(e)))
    missing = [cid for cid in batch if cid not in returned]
    if missing:
        failed_synonym_batches.append((missing, 'missing in batch response'))
    time.sleep(0.1)
    if batch_no % 10 == 0:
        print(f'Synonym batches: {min(batch_no * synonym_batch_size, len(unique_cids))}/{len(unique_cids)}', end='\r')

synonym_retry_cids = sorted({cid for batch, _ in failed_synonym_batches for cid in batch})
for cid in synonym_retry_cids:
    try:
        syn_url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON'
        r = session.get(syn_url, timeout=20)
        if r.status_code == 200:
            info_list = r.json().get('InformationList', {}).get('Information', [])
            if info_list:
                seen = set()
                cleaned = []
                for syn in info_list[0].get('Synonym', []):
                    syn = syn.strip()
                    syn_norm = syn.lower()
                    if not syn:
                        continue
                    if syn_norm.startswith('inchi='):
                        continue
                    if inchikey_pattern.fullmatch(syn):
                        continue
                    if syn_norm in seen:
                        continue
                    seen.add(syn_norm)
                    cleaned.append(syn)
                    if len(cleaned) == 10:
                        break
                cid_to_synonym_pool[cid] = cleaned
    except Exception:
        pass
    time.sleep(0.05)

for cid, indexes in cid_to_indexes.items():
    formula = cid_to_formula.get(cid, '')
    inchikey = cid_to_inchikey.get(cid, '')
    res_df.loc[indexes, 'molecular_formula'] = formula
    res_df.loc[indexes, 'inchikey'] = inchikey
    synonym_pool = cid_to_synonym_pool.get(cid, [])
    for idx in indexes:
        preferred_norm = str(res_df.loc[idx, 'preferred_name']).lower().strip()
        row_synonyms = []
        seen = set()
        for syn in synonym_pool:
            syn_norm = syn.lower().strip()
            if syn_norm == preferred_norm:
                continue
            if syn_norm in seen:
                continue
            seen.add(syn_norm)
            row_synonyms.append(syn)
            if len(row_synonyms) == 10:
                break
        res_df.loc[idx, 'synonyms'] = '|'.join(row_synonyms)

unresolved_property_cids = sorted([cid for cid in unique_cids if cid not in cid_to_formula or cid not in cid_to_inchikey])
unresolved_synonym_cids = sorted([cid for cid in unique_cids if cid not in cid_to_synonym_pool])
print(f'Formula filled: {sum(res_df["molecular_formula"] != "")}')
print(f'InChIKey filled: {sum(res_df["inchikey"] != "")}')
print(f'Synonyms filled: {sum(res_df["synonyms"] != "")}')
if failed_property_batches:
    print(f'Property retries needed for {len(property_retry_cids)} CIDs')
if failed_synonym_batches:
    print(f'Synonym retries needed for {len(synonym_retry_cids)} CIDs')
if unresolved_property_cids:
    print(f'WARNING: unresolved property CIDs after retry: {len(unresolved_property_cids)}')
    print(unresolved_property_cids[:20])
if unresolved_synonym_cids:
    print(f'WARNING: unresolved synonym CIDs after retry: {len(unresolved_synonym_cids)}')
    print(unresolved_synonym_cids[:20])

## Step 6: Save

In [ ]:
res_df.to_csv('chemical_enriched.csv', index=False)
print('Saved: chemical_enriched.csv')
files.download('chemical_enriched.csv')
import shutil
shutil.rmtree('pubchem', ignore_errors=True)
print('PubChem files cleaned up after final save.')